[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/part_07_research_capstone/ch35_packaging_deployment_reproducibility/03_coding_exam.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `INTERACTIVE STUDENT NOTEBOOK`


# Chapter 35 Coding Exam — Packaging, Deployment & Reproducible Research Releases (100 Points)

**Book Reference:** Chapter 35, Section 35.7 (Coding Exam)  
**Suggested Time:** 50–60 minutes (Open Book, Closed Solution)

| Task | Reproducible Research Deliverable | Points |
| :--- | :--- | :---: |
| **Task 35.1** | `batch_inference_with_quarantine(service, payloads)` (Resilient batch scoring + dead-letter quarantine) | 30 |
| **Task 35.2** | `benchmark_inference_latency(service, valid_payload, n_runs=50)` (Mean / p95 / max ms & req/s SLA) | 35 |
| **Task 35.3** | `package_end_to_end_research_release(mch_df, out_dir, seed=42)` (Full model + card + manifest + SHA-256 release) | 35 |
| **TOTAL** | | **100** |

In [ ]:
# @title Step 0: Click Play (▶) to Activate Chapter 35 Exam Grader & Helpers
!pip install -q "git+https://github.com/sharukh01621/first-line-to-first-paper.git#subdirectory=sharukh_tutor"
from datetime import datetime, timezone
import hashlib
import json
import math
from pathlib import Path
import platform
import time
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import sharukh_tutor as tutor


def generate_model_card_markdown(meta):
    """Provided helper from To-Do 35.3 for Task 35.3."""
    return (
        f"# Model Card: {meta['name']}\n\n"
        f"## Model Details\n"
        f"- **Name:** {meta['name']}\n"
        f"- **Version:** {meta['version']}\n\n"
        f"## Intended Use\n{meta['intended_use']}\n\n"
        f"## Training Data\n{meta['dataset']}\n\n"
        f"## Evaluation Metrics\n"
        f"- **ROC-AUC:** {float(meta['roc_auc']):.4f}\n"
        f"- **Brier Score:** {float(meta['brier_score']):.4f}\n\n"
        f"## Fairness & Limitations\n{meta['limitations']}\n"
    )


def build_reproducibility_bundle(artifact_path, config_dict, out_dir):
    """Provided helper from To-Do 35.4 for Task 35.3."""
    artifact, out = Path(artifact_path), Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    sha = hashlib.sha256(artifact.read_bytes()).hexdigest()
    manifest = {
        "artifact_name": artifact.name,
        "artifact_bytes": int(artifact.stat().st_size),
        "artifact_sha256": sha,
        "python_version": platform.python_version(),
        "created_utc": datetime.now(timezone.utc).strftime(
            "%Y-%m-%dT%H:%M:%SZ"
        ),
    }
    (out / "config.json").write_text(
        json.dumps(config_dict, indent=2), encoding="utf-8"
    )
    (out / "manifest.json").write_text(
        json.dumps(manifest, indent=2), encoding="utf-8"
    )
    return manifest


def verify_reproducibility_bundle(out_dir, artifact_path):
    """Provided helper from To-Do 35.5 for Task 35.3."""
    out = Path(out_dir)
    manifest = json.loads(
        (out / "manifest.json").read_text(encoding="utf-8")
    )
    sha = hashlib.sha256(Path(artifact_path).read_bytes()).hexdigest()
    ok_hash = sha == manifest["artifact_sha256"]
    ok_cfg = (out / "config.json").exists()
    return {
        "hash_match": bool(ok_hash),
        "config_present": bool(ok_cfg),
        "bundle_valid": bool(ok_hash and ok_cfg),
    }


tutor.activate(chapter="ch35", notebook="coding_exam")


---
## Exam Task 35.1 (30 Points): `batch_inference_with_quarantine(service, payloads)`

Implement resilient batch scoring with a dead-letter quarantine queue:
1. Iterate `for idx, p in enumerate(payloads):` without mutating `payloads`.
2. Call `service.predict_one(p)` inside `try / except ValueError as exc:`.
3. Append valid prediction dicts to `"succeeded"` and `{"index": idx, "error": str(exc)}` to `"quarantined"`.
4. Compute `success_rate = round(len(succeeded) / len(payloads), 4)` (or `0.0` if `payloads` is empty).
5. Return `{"succeeded": list, "quarantined": list, "success_rate": float}`.

In [ ]:
def batch_inference_with_quarantine(service, payloads):
    """Score valid payloads and quarantine ValueError rows by index."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace with your solution")


---
## Exam Task 35.2 (35 Points): `benchmark_inference_latency(service, valid_payload, n_runs=50)`

Benchmark single-request inference latency for clinical SLA reporting:
1. Run `service.predict_one(valid_payload)` exactly `n_runs` times, timing each call with `time.perf_counter()` in milliseconds (`* 1000.0`).
2. Compute `mean_ms`, `p95_ms` (`np.percentile(times_ms, 95)`), `max_ms`, and `throughput_per_sec = 1000.0 / mean_ms`.
3. Return `{"n_runs": int, "mean_ms": round4, "p95_ms": round4, "max_ms": round4, "throughput_per_sec": round2}`.

In [ ]:
def benchmark_inference_latency(service, valid_payload, n_runs=50):
    """Measure mean, p95, max latency (ms) & throughput (req/s)."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace with your solution")


---
## Exam Task 35.3 (35 Points): `package_end_to_end_research_release(mch_df, out_dir, seed=42)`

Without mutating `mch_df`, package a complete, peer-review-ready ML release inside `out_dir`:
1. Copy `mch_df`, normalize `residence` and `wealth_quintile` via `.astype(str).str.strip().str.title()`, and replace sentinel `999` with `np.nan` in `num_cols = ["maternal_age_years", "anc_visits", "maternal_hb_g_dl"]`.
2. Run a stratified 75/25 `train_test_split` (`random_state=seed`) predicting `child_stunted` from `num_cols + ["residence", "wealth_quintile"]`.
3. Fit a standard Scikit-Learn `Pipeline([("prep", ColumnTransformer(...)), ("model", LogisticRegression(max_iter=500))])` (median+scaler numeric, most_frequent+OneHotEncoder(`handle_unknown="ignore", sparse_output=False`) categorical).
4. Compute `test_auc` and `brier`, then write `model.joblib`, `MODEL_CARD.md`, `config.json`, and `manifest.json` inside `out_dir`.
5. Run `verify_reproducibility_bundle(out_dir, model_path)` and return `{"bundle_valid": bool, "test_auc": round4, "files_created": sorted_list}`.

In [ ]:
def package_end_to_end_research_release(mch_df, out_dir, seed=42):
    """Train, package, hash & verify a full MCH research release."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace with your solution")


In [ ]:
exam_submissions = {
    "ch35_exam_01": batch_inference_with_quarantine,
    "ch35_exam_02": benchmark_inference_latency,
    "ch35_exam_03": package_end_to_end_research_release,
}

tutor.check_exam("ch35", exam_submissions)
